# Modelo de Recomendación de Alcance — 1. Creación de variables

**Objetivo:** para cada NIC, estimar con qué alcance de brigada (canasta) es más probable que una
**suspensión** salga efectiva, aprendiendo del histórico. Este notebook **no entrena nada**: solo
construye el universo y las variables. El entrenamiento va en `2_Modelo_Alcance.ipynb`.

Sigue las mismas reglas y utilidades del modelo de riesgo social (`../Recomendacion Social/`).

## Las reglas de este notebook

**1. Una fila = una suspensión gestionada** (TO501 / TO504). Las tasas se miden solo sobre
suspensiones: una reconexión sale efectiva ~98 % de las veces y mezclarla premiaría al alcance que
más reconecta, no al que mejor suspende. El *historial* del NIC sí usa todas sus órdenes.

**2. Solo variables conocidas ANTES de la orden.** `av_resultado`, `estado_norm` y la subacción son
el resultado: entran únicamente como acumulados del pasado (`acumulado_previo`, `.shift(1)`).

**3. El universo es el mismo de la vista `dbanalitica.recomendacion_aire_scr`**: `av_resultado` no
nulo, sin `ON210`/`ON302` y sin anuladas. Así el modelo aprende de las mismas órdenes que la vista
con la que luego se va a predecir.

**4. El alcance (`CANASTA`) es una variable más.** En la fase 3 se le pregunta al modelo por cada
canasta para el mismo NIC y gana la de mayor probabilidad. Por eso hay variables de historial
*por canasta* (¿este NIC ya fue suspendido con esta canasta?).

## Homologación de alcances (acordada tras la reunión)

| Canasta | `brigada_homologada` |
|---|---|
| Pesada MT-AT | Brigada Pesada MT-AT, Brigada Pesada/ MT AT |
| Pesada Disponible | Pesada Disponible, Scr Pesada Disponibi |
| (D) Brigada Pesada | la pesada **nocturna**: va aparte, no se fusiona con Brigada Pesada |
| Fuera | Mtto, Web (una sola orden), SIN CLASIFICAR |


## 1. Librerías y conexión

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score
from sqlalchemy import create_engine

# La preparación del histórico vive en comun.py porque el notebook 3 la usa igual para predecir.
from comun import (CANASTAS, ENTORNOS, ENTORNOS_CANASTA, ENTORNOS_TECNICO, K_ENTORNO, VECINOS_TRAFO,
                   acumulado_previo, preparar_historia, variables_contexto)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)

DIR_DATOS = 'datos'
os.makedirs(DIR_DATOS, exist_ok=True)

load_dotenv('.env')
SCR_DATABASE_URL = os.getenv('SCR_DATABASE_URL')
if not SCR_DATABASE_URL:
    raise RuntimeError('Falta SCR_DATABASE_URL: copia el .env de "Recomendacion Social" a esta carpeta.')
engine = create_engine(SCR_DATABASE_URL)

### 1.1 Descarga de `historico_mo`

Solo las columnas necesarias y ya con el filtro de la vista. Se guarda en `parquet`: la próxima vez
lee del disco. **Borra el `.parquet` para refrescar** con los cierres nuevos.

In [ ]:
CACHE = f'{DIR_DATOS}/historico_mo_gestionadas.parquet'

CONSULTA = """
SELECT id, orden, nic, fecha_cierre, hora_fin, tipo_os, brigada_homologada,
       territorio, zona, municipio, localidad_barrio, tarifa, deuda_act, cant_factura_act,
       av_resultado, estado_norm, subaccion_homologada, id_transformador, id_circuito, id_tecnico
FROM dbanalitica.historico_mo
WHERE av_resultado IS NOT NULL
  AND av_resultado NOT IN ('ON210', 'ON302')
  AND estado_osf IS DISTINCT FROM 'Anulada'
"""

# Un caché de antes de agregar columnas a la consulta se descarta solo, en vez de fallar más abajo.
NUEVAS = ['id_transformador', 'id_circuito', 'id_tecnico']
df = pd.read_parquet(CACHE) if os.path.exists(CACHE) else None
if df is not None and set(NUEVAS) <= set(df.columns):
    print('leído de caché local')
else:
    t0 = time.time()
    df = pd.read_sql(CONSULTA, engine)
    df.to_parquet(CACHE, index=False)
    print(f'descargado de la BD en {time.time() - t0:.0f}s')

print(df.shape)

## 2. Preparación del histórico (`comun.preparar_historia`)

Normaliza el texto (mismo `norm_txt` del modelo de riesgo social), homologa la canasta, ordena por
NIC y fecha —orden **obligatorio**: todo el historial depende de él— y marca el resultado de cada
orden. Las marcas que empiezan con `_` salen del resultado de la propia orden: se usan solo para
construir acumulados del pasado y **se borran antes de guardar**.

Está en `comun.py` y no aquí porque el notebook 3 tiene que preparar el histórico exactamente igual
para predecir.

In [ ]:
df = preparar_historia(df)

# Lista blanca: un alcance nuevo que aparezca en la base queda fuera, y aquí se ve cuál fue.
# Si sale algo distinto de MTTO, WEB y SIN CLASIFICAR, hay que decidir dónde va (comun.CANASTAS).
print('Fuera de la homologación:')
print(df.loc[df['CANASTA'].isna(), 'brigada_homologada'].value_counts(dropna=False))
print()
print(df['CANASTA'].value_counts())
print()
print('rango:', df['fecha_cierre'].min().date(), '->', df['fecha_cierre'].max().date())
print('NICs:', df['nic'].nunique(), '| órdenes gestionadas:', len(df))

## 4. Historial del NIC

Mismo ladrillo del modelo de riesgo social: `acumulado_del_grupo − valor_de_la_fila`, para que cada
fila vea solo su pasado. Se calcula sobre **todas** las órdenes gestionadas del NIC, no solo las
suspensiones: una reconexión o una revisión también dicen algo del cliente.

In [ ]:
g = df.groupby('nic', sort=False)
df['ORDENES_PREVIAS'] = g.cumcount()

DERIVADAS = [
    ('_EFECTIVA',       'EFECTIVAS_PREVIAS'),
    ('_FALLIDA',        'FALLIDAS_PREVIAS'),
    ('_PERDIDA',        'PERDIDAS_PREVIAS'),
    ('_IMPOSIB',        'IMPOSIB_PREVIAS'),
    ('_PAGO_CLIENTE',   'PAGOS_CLIENTE_PREVIOS'),
    ('_RESISTENCIA',    'RESISTENCIAS_PREVIAS'),
    ('ES_SUSPENSION',   'SUSP_PREVIAS'),
    ('_SUSP_EFECTIVA',  'SUSP_EFECTIVAS_PREVIAS'),
]
for origen, destino in DERIVADAS:
    df[destino] = acumulado_previo(df, 'nic', origen)

# Sin historia -> NaN, no cero: cero diría que nunca le funcionó nada.
df['TASA_EF_HIST'] = (df['EFECTIVAS_PREVIAS'] / df['ORDENES_PREVIAS']).where(df['ORDENES_PREVIAS'] > 0)
df['TASA_EF_SUSP_HIST'] = (df['SUSP_EFECTIVAS_PREVIAS'] / df['SUSP_PREVIAS']).where(df['SUSP_PREVIAS'] > 0)

In [ ]:
# La orden inmediatamente anterior: cómo terminó (la "causal") y con qué alcance.
df['RESULTADO_ANTERIOR'] = g['av_resultado'].shift(1)
df['TIPO_OS_ANTERIOR'] = g['tipo_os'].shift(1)
df['CANASTA_ANTERIOR'] = g['CANASTA'].shift(1)

# Días desde la orden anterior. Más de 90 días sin orden = el NIC venía pagando, y en la reunión
# se acordó que eso debería bajar la probabilidad (se comprueba en el chequeo 4).
df['DIAS_DESDE_ANTERIOR'] = (df['fecha_cierre'] - g['fecha_cierre'].shift(1)).dt.days
df['SIN_ORDEN_90D'] = (df['DIAS_DESDE_ANTERIOR'] > 90).astype(float).where(df['DIAS_DESDE_ANTERIOR'].notna())

df['DEUDA_PREV'] = g['deuda_act'].shift(1)
df['DELTA_DEUDA'] = df['deuda_act'] - df['DEUDA_PREV']

### 4.1 Historial del NIC *con esa misma canasta*

Es la variable que más directamente responde la pregunta del negocio: ¿a este NIC ya se le fue con
esta canasta y funcionó? En la fase 3 se recalcula para cada canasta candidata.

In [ ]:
con_canasta = df['CANASTA'].notna()
dc = df.loc[con_canasta]

df.loc[con_canasta, 'ORDENES_PREVIAS_CANASTA'] = dc.groupby(['nic', 'CANASTA'], sort=False).cumcount()
df.loc[con_canasta, 'EF_PREVIAS_CANASTA'] = acumulado_previo(dc, ['nic', 'CANASTA'], '_EFECTIVA')
df['TASA_EF_CANASTA_NIC'] = (df['EF_PREVIAS_CANASTA'] / df['ORDENES_PREVIAS_CANASTA']).where(
    df['ORDENES_PREVIAS_CANASTA'] > 0)

## 5. Universo de entrenamiento y contexto de la orden

Suspensiones con canasta homologada y estado conocido (Efectiva, Fallida o Perdida). `SIN
CLASIFICAR` sale porque no se sabe si se cumplió.

In [ ]:
u = df[(df['ES_SUSPENSION'] == 1)
       & df['CANASTA'].notna()
       & df['estado_norm'].isin(['EFECTIVA', 'FALLIDA', 'PERDIDA'])].copy()

u['EFECTIVA'] = u['_EFECTIVA']
u['PERIODO'] = u['fecha_cierre'].dt.to_period('M').astype(str)

u = variables_contexto(u)   # segmento, estrato y deuda

# Sin variables de mes ni día: hay menos de un año de historia, así que el mes se confunde con la
# tendencia, y al recomendar no se sabe qué día va a salir la orden.

print(f'Suspensiones de entrenamiento: {len(u):,}  |  tasa efectiva: {u["EFECTIVA"].mean():.1%}')
u.groupby('PERIODO')['EFECTIVA'].agg(ordenes='size', tasa='mean').assign(tasa=lambda d: (d.tasa * 100).round(1))

## 6. Variables de entorno (barrio, municipio, transformador, técnico)

Se recorre el tiempo **calendario** (no el orden por NIC) para no mezclar el futuro de un vecino
con el pasado de otro. Suavizado bayesiano igual que en riesgo social: `(efectivas + K·prior) /
(órdenes + K)`, con `K = 20`. `TASA_EF_BARRIO_CANASTA` es la que le dice al modelo qué alcance
funciona en cada barrio.

El transformador, sus vecinos y el técnico se agregaron después de medir con octubre de 2026
(entrenando hasta septiembre): el acierto pasó de 68,1 % a 70,3 %. El técnico fue la que más
aportó, pero solo se conoce con la orden ya asignada; la matriz usa el técnico típico de cada
canasta. Se probaron y **se descartaron** las subacciones previas, la caracterización del predio
(+0,1) y el hábito de pago de `historico_balanza` (+0,4, a cambio de una descarga de 20 millones de
filas en cada corrida).

In [ ]:
u = u.sort_values(['fecha_cierre', '_HORA_FIN', 'id'], na_position='first').reset_index(drop=True)

def tasa_entorno(frame, key, nombre, col='EFECTIVA', destino=None):
    """Tasa histórica de `col` en el grupo `key`, suavizada y sin la fila actual."""
    prior = frame[col].mean()
    n_previas = frame.groupby(key, sort=False).cumcount()
    previas = acumulado_previo(frame, key, col)
    if destino is None:
        frame[f'ORDENES_PREV_{nombre}'] = n_previas
        destino = f'TASA_EF_{nombre}'
    frame[destino] = (previas + K_ENTORNO * prior) / (n_previas + K_ENTORNO)


# Las listas viven en comun.py: puntuar_canastas tiene que armar exactamente las mismas.
for key, nombre in ENTORNOS + ENTORNOS_CANASTA + ENTORNOS_TECNICO:
    tasa_entorno(u, key, nombre)
for marca, destino in VECINOS_TRAFO.items():
    tasa_entorno(u, 'id_transformador', None, col=marca, destino=destino)

### 6.1 Propensión de pago (servicio externo)

Las dos probabilidades del **Modelo Propensión de Pago Service** (SCR): que el cliente pague por su
cuenta y que pague ante la orden de suspensión. El usuario pidió que la matriz las use **sí o sí**,
y que nunca se contradigan: a más propensión de pago, menos probabilidad de suspender (notebook 2
lo fuerza con una restricción monótona).

El servicio solo responde por hoy, y entrenar con la propensión de hoy sería trampa. `propension.py`
corre **el mismo código y los mismos modelos del servicio** sobre los datos recortados al último día
del mes anterior a cada suspensión. Se comprobó contra el servicio en producción: 399 de 400 NICs
dieron el mismo número.

Medido con octubre de 2026 (entrenando hasta septiembre): el acierto no cambió (69,9 %) y el AUC
subió de 0,767 a 0,770. Aporta poco porque el modelo ya tenía el historial de pagos en la visita;
se deja porque es requisito del negocio.

In [ ]:
from propension import datos_servicio, propension_historica

CACHE_PROP = f'{DIR_DATOS}/propension_historica.parquet'
PROPENSION = ['PROB_PAGO_SIN_INTERVENCION', 'PROB_PAGO_CON_INTERVENCION']
pedidos = u[['nic', 'PERIODO']].drop_duplicates()
prop = pd.read_parquet(CACHE_PROP) if os.path.exists(CACHE_PROP) else pedidos.iloc[:0].assign(
    **{c: np.nan for c in PROPENSION})
# El mes en curso se recalcula siempre: le siguen llegando suspensiones.
mes_actual = u['PERIODO'].max()
prop = prop[prop['PERIODO'] != mes_actual]
faltan = pedidos.merge(prop[['nic', 'PERIODO']], how='left', indicator=True).query('_merge == "left_only"')
if len(faltan):
    t0 = time.time()
    balanza, mo = datos_servicio(engine, faltan['nic'].unique().tolist(), DIR_DATOS)
    prop = pd.concat([prop, propension_historica(balanza, mo, faltan)], ignore_index=True)
    prop.to_parquet(CACHE_PROP, index=False)
    print(f'propensión calculada para {len(faltan):,} (nic, mes) en {time.time() - t0:.0f}s')

u = u.merge(prop, on=['nic', 'PERIODO'], how='left')
print('sin propensión (el servicio no tenía datos):', f'{u[PROPENSION[1]].isna().mean():.2%}')
# La regla del negocio tiene que verse en los datos antes de forzarla en el modelo.
print(u.groupby(pd.qcut(u[PROPENSION[1]], 5, duplicates='drop'), observed=True)['EFECTIVA']
      .agg(efectividad='mean', n='size').round(3))

## 7. Validación de las variables

Los tres primeros chequeos detectan fugas; el cuarto muestra si hay señal; el quinto, si el alcance
se asigna al azar (no lo hace, y eso hay que tenerlo presente al leer la matriz).

In [ ]:
ok1 = ((u['EFECTIVAS_PREVIAS'] <= u['ORDENES_PREVIAS']).all()
       and (u['EF_PREVIAS_CANASTA'] <= u['ORDENES_PREVIAS_CANASTA']).all())

primera = u['ORDENES_PREVIAS'] == 0
ok2 = (u.loc[primera, 'EFECTIVAS_PREVIAS'].max() == 0
       and u.loc[primera, 'TASA_EF_HIST'].isna().all()
       and u.loc[primera, 'RESULTADO_ANTERIOR'].isna().all())

ok3 = (u['EFECTIVA'] != u['SUSP_EFECTIVAS_PREVIAS']).any()

print('1. previas <= totales            :', ok1)
print('2. primera orden sin historia    :', ok2)
print('3. sin identidad con el target   :', ok3)
assert ok1 and ok2 and ok3, 'Revisa la construcción: hay fuga de datos'

In [ ]:
def tasa(col, frame=u):
    return (frame.groupby(col, dropna=False, observed=True)['EFECTIVA']
                 .agg(tasa='mean', n='size').assign(tasa=lambda d: (d.tasa * 100).round(1)))


print('--- Más de 90 días sin orden (el "pagador") ---')
print(tasa('SIN_ORDEN_90D'))
print('\n--- Cómo terminó la orden anterior (las 10 más comunes) ---')
print(tasa('RESULTADO_ANTERIOR').sort_values('n', ascending=False).head(10))
print('\n--- Efectividad previa del NIC con la MISMA canasta ---')
print(tasa(pd.cut(u['TASA_EF_CANASTA_NIC'], [-0.01, 0, 0.5, 0.99, 1])))
print('\n--- Quintil de la tasa barrio x canasta ---')
print(tasa(pd.qcut(u['TASA_EF_BARRIO_CANASTA'], 5, duplicates='drop')))

In [ ]:
# CHEQUEO 5 - el alcance NO se asigna al azar.
# Si las canastas atienden clientes distintos, parte de su "éxito" es del cliente y no de la canasta.
sesgo = u.groupby('CANASTA').agg(
    ordenes=('EFECTIVA', 'size'),
    pct_efectiva=('EFECTIVA', 'mean'),
    deuda_mediana=('deuda_act', 'median'),
    pct_sin_historia=('ORDENES_PREVIAS', lambda s: (s == 0).mean()),
    pct_con_imposib_previa=('IMPOSIB_PREVIAS', lambda s: (s > 0).mean()),
    pct_pagador_90d=('SIN_ORDEN_90D', 'mean'),
).sort_values('ordenes', ascending=False)
for c in ['pct_efectiva', 'pct_sin_historia', 'pct_con_imposib_previa', 'pct_pagador_90d']:
    sesgo[c] = (sesgo[c] * 100).round(1)
sesgo

In [ ]:
# ¿La mejor canasta cambia según el municipio? Si siempre gana la misma, el modelo tampoco
# va a poder recomendar otra cosa. Solo combinaciones con al menos 200 suspensiones.
mc = u.groupby(['municipio', 'CANASTA'])['EFECTIVA'].agg(['mean', 'size'])
mc = mc[mc['size'] >= 200].reset_index()
mejor = mc.loc[mc.groupby('municipio')['mean'].idxmax()]
print('Canasta con mayor efectividad, por municipio:')
print(mejor['CANASTA'].value_counts())

## 8. Organización y guardado

Esta lista es el contrato con el notebook 2: lo que esté en `PROHIBIDAS` no entra al modelo.

In [ ]:
IDENTIFICADORES = ['id', 'orden', 'nic', 'fecha_cierre', 'PERIODO']

CATEGORICAS = [
    'CANASTA', 'tipo_os', 'territorio', 'zona', 'municipio', 'localidad_barrio', 'SEGMENTO',
    'RESULTADO_ANTERIOR', 'TIPO_OS_ANTERIOR', 'CANASTA_ANTERIOR',
]

NUMERICAS = [
    # contexto de la orden
    'ESTRATO', 'deuda_act', 'LOG_DEUDA', 'cant_factura_act', 'DEUDA_POR_FACTURA',
    # historial del NIC
    'ORDENES_PREVIAS', 'EFECTIVAS_PREVIAS', 'FALLIDAS_PREVIAS', 'PERDIDAS_PREVIAS',
    'IMPOSIB_PREVIAS', 'PAGOS_CLIENTE_PREVIOS', 'RESISTENCIAS_PREVIAS',
    'SUSP_PREVIAS', 'SUSP_EFECTIVAS_PREVIAS', 'TASA_EF_HIST', 'TASA_EF_SUSP_HIST',
    'DIAS_DESDE_ANTERIOR', 'SIN_ORDEN_90D', 'DELTA_DEUDA',
    # historial del NIC con esta canasta
    'ORDENES_PREVIAS_CANASTA', 'EF_PREVIAS_CANASTA', 'TASA_EF_CANASTA_NIC',
    # entorno
    'ORDENES_PREV_BARRIO', 'TASA_EF_BARRIO', 'TASA_EF_MUNICIPIO',
    'ORDENES_PREV_BARRIO_CANASTA', 'TASA_EF_BARRIO_CANASTA', 'TASA_EF_MUNICIPIO_CANASTA',
    # red eléctrica: cómo les fue a los vecinos del mismo transformador y circuito
    'ORDENES_PREV_TRAFO', 'TASA_EF_TRAFO', 'TASA_EF_CIRCUITO',
    'ORDENES_PREV_TRAFO_CANASTA', 'TASA_EF_TRAFO_CANASTA',
    'TASA_AGRESIVO_TRAFO', 'TASA_IMPOSIB_TRAFO', 'TASA_PAGO_VISITA_TRAFO',
    # técnico: solo se sabe con la orden asignada; la matriz usa el típico de la canasta
    'ORDENES_PREV_TECNICO', 'TASA_EF_TECNICO', 'ORDENES_PREV_TECNICO_MUNICIPIO', 'TASA_EF_TECNICO_MUNICIPIO',
    # servicio de propensión de pago (6.1); el notebook 2 las fuerza a bajar la probabilidad
    *PROPENSION,
]

FEATURES = CATEGORICAS + NUMERICAS
TARGET = 'EFECTIVA'
# El resultado de la propia orden: sirve para analizar, nunca para predecir.
PROHIBIDAS = ['av_resultado', 'estado_norm', 'subaccion_homologada']

# CatBoost no acepta nulos en categóricas; en las de historial el nulo significa "primera orden".
for c in ['RESULTADO_ANTERIOR', 'TIPO_OS_ANTERIOR', 'CANASTA_ANTERIOR', 'SEGMENTO']:
    u[c] = u[c].fillna('SIN HISTORIA' if c != 'SEGMENTO' else 'SIN DATO')

In [ ]:
def barrido_fugas(frame, features, target=TARGET):
    """AUC de cada feature por sí sola. >0.90 = casi seguro fuga de datos."""
    y = frame[target]
    filas = []
    for f in features:
        s = frame[f]
        if pd.api.types.is_numeric_dtype(s):
            score = roc_auc_score(y, s.fillna(s.median()))
        else:
            score = roc_auc_score(y, y.groupby(s.fillna('NA')).transform('mean'))
        filas.append((f, max(score, 1 - score)))
    r = (pd.DataFrame(filas, columns=['feature', 'auc'])
           .sort_values('auc', ascending=False).reset_index(drop=True))
    r['alerta'] = np.where(r['auc'] > 0.90, 'FUGA?', '')
    return r


barrido = barrido_fugas(u, FEATURES)
print(barrido.head(15).round(4).to_string(index=False))

sospechosas = barrido.loc[barrido['auc'] > 0.90, 'feature'].tolist()
assert not sospechosas, f'Revisa estas variables antes de seguir: {sospechosas}'
print('\nSin fugas detectadas.')

In [ ]:
(u[NUMERICAS].isna().mean() * 100).round(1).rename('% nulos').sort_values(ascending=False).head(12).to_frame()

In [ ]:
df_universo = u[IDENTIFICADORES + FEATURES + [TARGET] + PROHIBIDAS].copy()
df_universo.to_parquet(f'{DIR_DATOS}/df_universo_alcance.parquet', index=False)

with open(f'{DIR_DATOS}/features_alcance.json', 'w', encoding='utf-8') as f:
    json.dump({'features': FEATURES, 'categoricas': CATEGORICAS, 'numericas': NUMERICAS,
               'target': TARGET, 'prohibidas': PROHIBIDAS, 'canastas': CANASTAS},
              f, indent=2, ensure_ascii=False)

print('guardado:', df_universo.shape)

## 9. Qué sigue (notebook 2)

1. **Split por tiempo**: entrenar con todo menos los últimos 2 meses y medir con esos meses.
2. **CatBoost** con `cat_features = CATEGORICAS`, sin `class_weights` (en riesgo social empeoró la calibración).
3. **AUC y calibración**: la matriz compara probabilidades entre canastas, así que tienen que estar bien calibradas, no solo bien ordenadas.
4. **Comparar contra una línea base** (la tasa barrio × canasta, la idea de la fórmula fija que se descartó): si el modelo no le gana, no aporta.

### Limitaciones que hay que decir de frente

- **El alcance no se asigna al azar** (chequeo 5). El modelo aprende "a suspensiones parecidas les fue
  mejor con X", que no es lo mismo que "mandar X causa el éxito".
- **Menos de un año de historia**: no hay estacionalidad anual y el historial por NIC es corto.
- **Algunas canastas tienen pocas órdenes** (Pesada Disponible, la nocturna): su probabilidad será
  menos confiable que la de Brigada Pesada.
